In [ ]:
from google.colab import files
uploaded = files.upload()

Saving clean_day1_all_probs.npz to clean_day1_all_probs.npz
Saving data_identification.csv to data_identification.csv
Saving emotion.csv to emotion.csv
Saving final_posts.json to final_posts.json
Saving sampleSubmission.csv to sampleSubmission.csv


In [ ]:
!nvidia-smi
!pip install -q -U transformers accelerate scikit-learn

Wed Nov 18 23:43:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
import os
import re
import json
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_cosine_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, classification_report
from scipy.sparse import hstack

# ==========================================
# 1. Reproducibility & Device
# ==========================================
SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ==========================================
# 2. Load Strictly In-Competition Data
# ==========================================
with open("final_posts.json", "r", encoding="utf-8") as f:
    raw_posts = json.load(f)

if isinstance(raw_posts, dict):
    if all(isinstance(v, str) for v in list(raw_posts.values())[:5]):
        records = [{"post_id": str(k), "text": v} for k, v in raw_posts.items()]
    else:
        first_val = next(iter(raw_posts.values()))
        raw_posts = first_val if isinstance(first_val, list) else list(raw_posts.values())

if isinstance(raw_posts, list):
    records = []
    for item in raw_posts:
        post = item["root"]["_source"]["post"] if "root" in item else item
        pid = post.get("post_id") or post.get("id")
        txt = post.get("text") or post.get("content") or ""
        records.append({"post_id": str(pid), "text": txt})

posts_df = pd.DataFrame(records)
ident_df = pd.read_csv("data_identification.csv", dtype=str)
emo_df   = pd.read_csv("emotion.csv", dtype=str)
full_df  = posts_df.merge(ident_df, on="post_id", how="inner")

def light_clean(text):
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", "http", text)
    text = re.sub(r"@\w+", "@user", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

full_df["clean_text"] = full_df["text"].apply(light_clean)
train_df = full_df[full_df["set"] == "train"].merge(emo_df, on="post_id", how="inner").reset_index(drop=True)
test_df  = full_df[full_df["set"] == "test"].reset_index(drop=True)

sample_sub = pd.read_csv("sampleSubmission.csv", dtype=str)
sub_id_col = sample_sub.columns[0]
test_df = sample_sub[[sub_id_col]].merge(test_df, left_on=sub_id_col, right_on="post_id", how="left")

le = LabelEncoder()
y_all = le.fit_transform(train_df["label"].astype(str))
classes = le.classes_
num_classes = len(classes)
counts = np.bincount(y_all)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# ==========================================
# 3. Model Family A: Clean Lexical Word+Char TF-IDF Calibrated Ensemble
# ==========================================
print("\n[1/3] Training Clean Lexical TF-IDF Calibrated Ensemble...")
word_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=30000, sublinear_tf=True)
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_features=45000, sublinear_tf=True)

X_tr_lex = hstack([
    normalize(word_vec.fit_transform(train_df["clean_text"])),
    normalize(char_vec.fit_transform(train_df["clean_text"])),
]).tocsr()
X_te_lex = hstack([
    normalize(word_vec.transform(test_df["clean_text"])),
    normalize(char_vec.transform(test_df["clean_text"])),
]).tocsr()

tempered_w = {c: (len(y_all) / (num_classes * counts[c])) ** 0.5 for c in range(num_classes)}
oof_lex = np.zeros((len(train_df), num_classes), dtype=np.float32)
test_lex = np.zeros((len(test_df), num_classes), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr_lex, y_all)):
    m_svc = CalibratedClassifierCV(LinearSVC(C=0.25, class_weight=tempered_w, random_state=SEED + fold), cv=3)
    m_lr  = LogisticRegression(C=1.5, class_weight=tempered_w, max_iter=1000, random_state=SEED + fold)
    m_svc.fit(X_tr_lex[tr_idx], y_all[tr_idx])
    m_lr.fit(X_tr_lex[tr_idx], y_all[tr_idx])
    oof_lex[va_idx] = 0.5 * m_svc.predict_proba(X_tr_lex[va_idx]) + 0.5 * m_lr.predict_proba(X_tr_lex[va_idx])
    test_lex += (0.5 * m_svc.predict_proba(X_te_lex) + 0.5 * m_lr.predict_proba(X_te_lex)) / 5.0

print(f"Clean Lexical TF-IDF 5-Fold CV Macro-F1: {f1_score(y_all, oof_lex.argmax(axis=1), average='macro'):.4f}")

# ==========================================
# 4. Custom Mean+Max Pooling + Multi-Sample Dropout Transformer Classifier
# ==========================================
class MeanMaxPoolClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.update({"hidden_dropout_prob": 0.1, "attention_probs_dropout_prob": 0.1})
        self.backbone = AutoModel.from_pretrained(model_name, config=self.config)
        hidden = self.config.hidden_size
        self.dropouts = nn.ModuleList([nn.Dropout(0.1 * i) for i in range(1, 6)])
        self.fc = nn.Linear(hidden * 2, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        mean_pool = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        max_pool = out.masked_fill(mask == 0, -1e4).max(dim=1).values
        feat = torch.cat([mean_pool, max_pool], dim=-1)
        logits = sum(self.fc(drop(feat)) for drop in self.dropouts) / len(self.dropouts)
        return logits

def train_clean_backbone(model_name, lr=2e-5, epochs=4, batch_size=32, max_len=144):
    print(f"\nTraining 5-Fold Clean Backbone: {model_name} (Pure MLM/RTD, zero emotion supervision)...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    class TextDS(Dataset):
        def __init__(self, texts, labels=None):
            self.enc = tokenizer(list(texts), truncation=True, max_length=max_len, padding=False)
            self.labels = list(labels) if labels is not None else None
        def __len__(self):
            return len(self.enc["input_ids"])
        def __getitem__(self, i):
            d = {"input_ids": self.enc["input_ids"][i], "attention_mask": self.enc["attention_mask"][i]}
            if self.labels is not None:
                d["labels"] = self.labels[i]
            return d

    def collate(batch):
        ml = max(len(x["input_ids"]) for x in batch)
        pad_id = tokenizer.pad_token_id or 0
        ids, masks, labs = [], [], []
        for x in batch:
            pl = ml - len(x["input_ids"])
            ids.append(x["input_ids"] + [pad_id] * pl)
            masks.append(x["attention_mask"] + [0] * pl)
            if "labels" in x:
                labs.append(x["labels"])
        out = {"input_ids": torch.tensor(ids, dtype=torch.long), "attention_mask": torch.tensor(masks, dtype=torch.long)}
        if labs:
            out["labels"] = torch.tensor(labs, dtype=torch.long)
        return out

    test_loader = DataLoader(TextDS(test_df["clean_text"]), batch_size=64, shuffle=False, collate_fn=collate)
    class_weights = torch.tensor([(len(y_all) / (num_classes * counts[c])) ** 0.45 for c in range(num_classes)], dtype=torch.float, device=device)

    oof_p = np.zeros((len(train_df), num_classes), dtype=np.float32)
    test_p = np.zeros((len(test_df), num_classes), dtype=np.float32)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, y_all), 1):
        t0 = time.time()
        seed_everything(SEED + fold * 10)
        tr_loader = DataLoader(TextDS(train_df.loc[tr_idx, "clean_text"], y_all[tr_idx]), batch_size=batch_size, shuffle=True, collate_fn=collate)
        va_loader = DataLoader(TextDS(train_df.loc[va_idx, "clean_text"], y_all[va_idx]), batch_size=64, shuffle=False, collate_fn=collate)

        model = MeanMaxPoolClassifier(model_name, num_classes).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        total_steps = len(tr_loader) * epochs
        scheduler = get_cosine_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
        # DeBERTa-v3 prefers bf16 on A100/L4 or fp32/fp16 carefully
        use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
        scaler = torch.amp.GradScaler("cuda", enabled=(not use_bf16 and device.type == "cuda"))

        best_f1, best_va, best_state = -1.0, None, None
        for ep in range(1, epochs + 1):
            model.train()
            for b in tr_loader:
                optimizer.zero_grad()
                with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=(device.type == "cuda")):
                    logits = model(b["input_ids"].to(device), b["attention_mask"].to(device))
                    loss = criterion(logits, b["labels"].to(device))
                if scaler.is_enabled():
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                scheduler.step()

            model.eval()
            va_list = []
            with torch.inference_mode():
                for b in va_loader:
                    with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=(device.type == "cuda")):
                        logits = model(b["input_ids"].to(device), b["attention_mask"].to(device))
                        va_list.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
            va_probs = np.vstack(va_list)
            f1_ep = f1_score(y_all[va_idx], va_probs.argmax(axis=1), average="macro")
            if f1_ep > best_f1:
                best_f1, best_va = f1_ep, va_probs
                best_state = {k: v.cpu() for k, v in model.state_dict().items()}

        oof_p[va_idx] = best_va
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
        model.eval()
        te_list = []
        with torch.inference_mode():
            for b in test_loader:
                with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=(device.type == "cuda")):
                    logits = model(b["input_ids"].to(device), b["attention_mask"].to(device))
                    te_list.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
        test_p += np.vstack(te_list) / 5.0
        print(f"  Fold {fold}/5 | Val Macro-F1: {best_f1:.4f} | {time.time() - t0:.1f}s")

    cv_f1 = f1_score(y_all, oof_p.argmax(axis=1), average="macro")
    print(f"--> {model_name} 5-Fold OOF Macro-F1: {cv_f1:.4f}")
    return oof_p, test_p

# Train two pure pre-trained backbones (zero external emotion labels)
oof_deb, test_deb = train_clean_backbone("microsoft/deberta-v3-base", lr=2.2e-5, epochs=4, batch_size=32)
oof_rob, test_rob = train_clean_backbone("FacebookAI/roberta-base", lr=2.5e-5, epochs=4, batch_size=32)

# ==========================================
# 5. Heterogeneous Ensemble + Nested OOF Class-Multiplier Tuning
# ==========================================
oof_ens = 0.50 * oof_deb + 0.35 * oof_rob + 0.15 * oof_lex
test_ens = 0.50 * test_deb + 0.35 * test_rob + 0.15 * test_lex
f1_ens_raw = f1_score(y_all, oof_ens.argmax(axis=1), average="macro")

# Nested OOF coordinate ascent for class multipliers (strictly leak-free)
def fit_multipliers(probs, labels, steps=2):
    mult = np.ones(num_classes, dtype=np.float32)
    best = f1_score(labels, (probs * mult).argmax(axis=1), average="macro")
    for _ in range(steps):
        for c in range(num_classes):
            best_m = mult[c]
            for cand in np.linspace(0.75, 1.35, 31):
                trial = mult.copy()
                trial[c] = cand
                s = f1_score(labels, (probs * trial).argmax(axis=1), average="macro")
                if s > best + 1e-5:
                    best, best_m = s, cand
            mult[c] = best_m
    return mult

oof_ens_opt = np.zeros(len(y_all), dtype=int)
fold_mults = []
for tr_idx, va_idx in skf.split(oof_ens, y_all):
    m = fit_multipliers(oof_ens[tr_idx], y_all[tr_idx])
    fold_mults.append(m)
    oof_ens_opt[va_idx] = (oof_ens[va_idx] * m).argmax(axis=1)

f1_ens_opt = f1_score(y_all, oof_ens_opt, average="macro")
mean_mult = np.mean(fold_mults, axis=0)

print("\n" + "=" * 74)
print("STRICTLY COMPLIANT HETEROGENEOUS ENSEMBLE SUMMARY")
print("=" * 74)
print(f"1. Clean Lexical TF-IDF Calibrated Ensemble  : {f1_score(y_all, oof_lex.argmax(axis=1), average='macro'):.4f}")
print(f"2. Clean 5-Fold RoBERTa-base (Mean+Max Pool) : {f1_score(y_all, oof_rob.argmax(axis=1), average='macro'):.4f}")
print(f"3. Clean 5-Fold DeBERTa-v3-base (Mean+Max)   : {f1_score(y_all, oof_deb.argmax(axis=1), average='macro'):.4f}")
print(f"4. 3-Family Heterogeneous Blend (Raw Argmax) : {f1_ens_raw:.4f}")
print(f"5. 3-Family Blend + Nested OOF Multipliers   : {f1_ens_opt:.4f}")
print("Learned Mean Class Multipliers:", dict(zip(classes, np.round(mean_mult, 3))))
print("=" * 74)

best_oof = oof_ens_opt if f1_ens_opt > f1_ens_raw else oof_ens.argmax(axis=1)
print(classification_report(train_df["label"], le.inverse_transform(best_oof), digits=4))

# Save both clean submissions in Kaggle's exact format (id, emotion)
sub_clean_v2 = pd.DataFrame({
    "id": test_df[sub_id_col],
    "emotion": le.inverse_transform(test_ens.argmax(axis=1))
})
sub_clean_v2.to_csv("submission_clean_heterogeneous_blend.csv", index=False)

sub_clean_v2_opt = pd.DataFrame({
    "id": test_df[sub_id_col],
    "emotion": le.inverse_transform((test_ens * mean_mult).argmax(axis=1))
})
sub_clean_v2_opt.to_csv("submission_clean_heterogeneous_opt.csv", index=False)

from google.colab import files
files.download("submission_clean_heterogeneous_blend.csv")
files.download("submission_clean_heterogeneous_opt.csv")

Using device: cuda

[1/3] Training Clean Lexical TF-IDF Calibrated Ensemble...
Clean Lexical TF-IDF 5-Fold CV Macro-F1: 0.5087

Training 5-Fold Clean Backbone: microsoft/deberta-v3-base (Pure MLM/RTD, zero emotion supervision)...


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  371MB            

... (34 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
from google.colab import files

df_clean = pd.read_csv("submission_clean_roberta_blend.csv")
df_clean.columns = ["id", "emotion"]
df_clean.to_csv("submission_clean_roberta_blend.csv", index=False)

print(df_clean.head(3))
files.download("submission_clean_roberta_blend.csv")

        id   emotion
0  xyzx9jp  surprise
1  l3vzzxt   sadness
2  g680iug   sadness


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ==========================================
# Fix DeBERTa-v3 (Explicit torch_dtype=torch.float32) & Blend with Saved RoBERTa + TF-IDF
# ==========================================
from transformers import AutoModelForSequenceClassification

def train_clean_deberta_fp32(model_name="microsoft/deberta-v3-base", lr=2e-5, epochs=4, batch_size=32, max_len=128):
    print(f"\nTraining 5-Fold {model_name} in explicit FP32...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    class TextDS(Dataset):
        def __init__(self, texts, labels=None):
            self.enc = tokenizer(list(texts), truncation=True, max_length=max_len, padding=False)
            self.labels = list(labels) if labels is not None else None
        def __len__(self):
            return len(self.enc["input_ids"])
        def __getitem__(self, i):
            d = {"input_ids": self.enc["input_ids"][i], "attention_mask": self.enc["attention_mask"][i]}
            if self.labels is not None:
                d["labels"] = self.labels[i]
            return d

    def collate(batch):
        ml = max(len(x["input_ids"]) for x in batch)
        pad_id = tokenizer.pad_token_id or 0
        ids, masks, labs = [], [], []
        for x in batch:
            pl = ml - len(x["input_ids"])
            ids.append(x["input_ids"] + [pad_id] * pl)
            masks.append(x["attention_mask"] + [0] * pl)
            if "labels" in x:
                labs.append(x["labels"])
        out = {"input_ids": torch.tensor(ids, dtype=torch.long), "attention_mask": torch.tensor(masks, dtype=torch.long)}
        if labs:
            out["labels"] = torch.tensor(labs, dtype=torch.long)
        return out

    test_loader = DataLoader(TextDS(test_df["clean_text"]), batch_size=64, shuffle=False, collate_fn=collate)
    class_weights = torch.tensor([(len(y_all) / (num_classes * counts[c])) ** 0.45 for c in range(num_classes)], dtype=torch.float32, device=device)

    oof_p = np.zeros((len(train_df), num_classes), dtype=np.float32)
    test_p = np.zeros((len(test_df), num_classes), dtype=np.float32)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, y_all), 1):
        t0 = time.time()
        seed_everything(SEED + fold * 10)
        tr_loader = DataLoader(TextDS(train_df.loc[tr_idx, "clean_text"], y_all[tr_idx]), batch_size=batch_size, shuffle=True, collate_fn=collate)
        va_loader = DataLoader(TextDS(train_df.loc[va_idx, "clean_text"], y_all[va_idx]), batch_size=64, shuffle=False, collate_fn=collate)

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=num_classes,
            hidden_dropout_prob=0.1,
            attention_probs_dropout_prob=0.1,
            torch_dtype=torch.float32
        ).float().to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        total_steps = len(tr_loader) * epochs
        scheduler = get_cosine_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)

        best_f1, best_va, best_state = -1.0, None, None
        for ep in range(1, epochs + 1):
            model.train()
            for b in tr_loader:
                optimizer.zero_grad()
                logits = model(input_ids=b["input_ids"].to(device), attention_mask=b["attention_mask"].to(device)).logits.float()
                loss = criterion(logits, b["labels"].to(device))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()

            model.eval()
            va_list = []
            with torch.inference_mode():
                for b in va_loader:
                    logits = model(input_ids=b["input_ids"].to(device), attention_mask=b["attention_mask"].to(device)).logits.float()
                    va_list.append(torch.softmax(logits, dim=-1).cpu().numpy())
            va_probs = np.vstack(va_list)
            f1_ep = f1_score(y_all[va_idx], va_probs.argmax(axis=1), average="macro")
            if f1_ep > best_f1:
                best_f1, best_va = f1_ep, va_probs
                best_state = {k: v.cpu() for k, v in model.state_dict().items()}

        oof_p[va_idx] = best_va
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
        model.eval()
        te_list = []
        with torch.inference_mode():
            for b in test_loader:
                logits = model(input_ids=b["input_ids"].to(device), attention_mask=b["attention_mask"].to(device)).logits.float()
                te_list.append(torch.softmax(logits, dim=-1).cpu().numpy())
        test_p += np.vstack(te_list) / 5.0
        print(f"  Fold {fold}/5 | Val Macro-F1: {best_f1:.4f} | {time.time() - t0:.1f}s")

    print(f"--> {model_name} (FP32) 5-Fold OOF Macro-F1: {f1_score(y_all, oof_p.argmax(axis=1), average='macro'):.4f}")
    return oof_p, test_p

oof_deb_ok, test_deb_ok = train_clean_deberta_fp32("microsoft/deberta-v3-base", lr=2e-5, epochs=4, batch_size=32)

# Grid-search optimal OOF blend weights across DeBERTa-v3 + RoBERTa-base + Lexical TF-IDF
best_w, best_blend_f1 = (0.5, 0.4, 0.1), -1.0
for wd in np.linspace(0.35, 0.65, 7):
    for wr in np.linspace(0.25, 0.55, 7):
        wl = round(1.0 - wd - wr, 4)
        if wl < 0:
            continue
        p = wd * oof_deb_ok + wr * oof_rob + wl * oof_lex
        s = f1_score(y_all, p.argmax(axis=1), average="macro")
        if s > best_blend_f1:
            best_blend_f1 = s
            best_w = (round(wd, 3), round(wr, 3), wl)

wd, wr, wl = best_w
oof_clean_tri = wd * oof_deb_ok + wr * oof_rob + wl * oof_lex
test_clean_tri = wd * test_deb_ok + wr * test_rob + wl * test_lex

# Nested OOF class-multiplier tuning
oof_tri_opt = np.zeros(len(y_all), dtype=int)
fold_mults = []
for tr_idx, va_idx in skf.split(oof_clean_tri, y_all):
    m = fit_multipliers(oof_clean_tri[tr_idx], y_all[tr_idx])
    fold_mults.append(m)
    oof_tri_opt[va_idx] = (oof_clean_tri[va_idx] * m).argmax(axis=1)

f1_tri_opt = f1_score(y_all, oof_tri_opt, average="macro")
mean_mult = np.mean(fold_mults, axis=0)

print("\n" + "=" * 74)
print("FIXED 100% COMPLIANT TRI-FAMILY ENSEMBLE SUMMARY")
print("=" * 74)
print(f"1. Clean Lexical TF-IDF Calibrated Ensemble  : {f1_score(y_all, oof_lex.argmax(axis=1), average='macro'):.4f}")
print(f"2. Clean 5-Fold RoBERTa-base (Mean+Max Pool) : {f1_score(y_all, oof_rob.argmax(axis=1), average='macro'):.4f}")
print(f"3. Clean 5-Fold DeBERTa-v3-base (FP32)       : {f1_score(y_all, oof_deb_ok.argmax(axis=1), average='macro'):.4f}")
print(f"4. Tri-Family Blend {best_w} (Raw)  : {best_blend_f1:.4f}")
print(f"5. Tri-Family Blend + Nested OOF Multipliers   : {f1_tri_opt:.4f}")
print("=" * 74)

sub_tri_raw = pd.DataFrame({"id": test_df[sub_id_col], "emotion": le.inverse_transform(test_clean_tri.argmax(axis=1))})
sub_tri_raw.to_csv("submission_clean_trifamily_blend.csv", index=False)

sub_tri_opt = pd.DataFrame({"id": test_df[sub_id_col], "emotion": le.inverse_transform((test_clean_tri * mean_mult).argmax(axis=1))})
sub_tri_opt.to_csv("submission_clean_trifamily_opt.csv", index=False)

from google.colab import files
files.download("submission_clean_trifamily_blend.csv")
files.download("submission_clean_trifamily_opt.csv")


Training 5-Fold microsoft/deberta-v3-base in explicit FP32...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.den

  Fold 1/5 | Val Macro-F1: 0.5339 | 55.3s


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

... (11 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score
from google.colab import files
import numpy as np
import pandas as pd

# 1. Extract pure general-purpose semantic embeddings (BAAI/bge-large-en-v1.5, MIT license, zero emotion labels)
bge_model = SentenceTransformer("BAAI/bge-large-en-v1.5", device=str(device))
X_tr_bge = normalize(bge_model.encode(train_df["clean_text"].tolist(), batch_size=64, show_progress_bar=False))
X_te_bge = normalize(bge_model.encode(test_df["clean_text"].tolist(), batch_size=64, show_progress_bar=False))

tempered_w = {c: (len(y_all) / (num_classes * counts[c])) ** 0.45 for c in range(num_classes)}
oof_bge = np.zeros((len(train_df), num_classes), dtype=np.float32)
test_bge = np.zeros((len(test_df), num_classes), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr_bge, y_all)):
    m_svc = CalibratedClassifierCV(LinearSVC(C=0.35, class_weight=tempered_w, random_state=SEED + fold), cv=3)
    m_lr  = LogisticRegression(C=2.0, class_weight=tempered_w, max_iter=1000, random_state=SEED + fold)
    m_svc.fit(X_tr_bge[tr_idx], y_all[tr_idx])
    m_lr.fit(X_tr_bge[tr_idx], y_all[tr_idx])
    oof_bge[va_idx] = 0.5 * m_svc.predict_proba(X_tr_bge[va_idx]) + 0.5 * m_lr.predict_proba(X_tr_bge[va_idx])
    test_bge += (0.5 * m_svc.predict_proba(X_te_bge) + 0.5 * m_lr.predict_proba(X_te_bge)) / 5.0

print(f"Clean BGE-Large-v1.5 5-Fold CV Macro-F1: {f1_score(y_all, oof_bge.argmax(axis=1), average='macro'):.4f}")

# 2. Pure Raw Soft-Voting Blend (No threshold/multiplier overfitting!)
# Combines 4 distinct clean model families: RoBERTa-base + DeBERTa-v3-base + BGE-Large + Lexical TF-IDF
weights_list = [
    ("Balanced_4Family", (0.35, 0.25, 0.25, 0.15)),  # (RoBERTa, DeBERTa, BGE-Large, Lexical)
    ("DenseHeavy_4Family", (0.38, 0.27, 0.25, 0.10)),
]

for name, (wr, wd, wb, wl) in weights_list:
    oof_4f = wr * oof_rob + wd * oof_deb_ok + wb * oof_bge + wl * oof_lex
    f1_4f = f1_score(y_all, oof_4f.argmax(axis=1), average="macro")
    print(f"{name} {(wr, wd, wb, wl)} -> 5-Fold OOF Raw Macro-F1: {f1_4f:.4f}")

# Pick the best raw blend on OOF
best_4f_name, (wr, wd, wb, wl) = max(
    weights_list,
    key=lambda item: f1_score(
        y_all,
        (item[1][0]*oof_rob + item[1][1]*oof_deb_ok + item[1][2]*oof_bge + item[1][3]*oof_lex).argmax(axis=1),
        average="macro"
    )
)

test_4f = wr * test_rob + wd * test_deb_ok + wb * test_bge + wl * test_lex
sub_4f = pd.DataFrame({
    "id": test_df[sub_id_col],
    "emotion": le.inverse_transform(test_4f.argmax(axis=1))
})
sub_4f.to_csv("submission_clean_4family_blend.csv", index=False)

# Save all Day 1 clean OOF/test arrays for tomorrow's Large-model Grand Ensemble
np.savez(
    "clean_day1_all_probs.npz",
    oof_rob=oof_rob, test_rob=test_rob,
    oof_deb=oof_deb_ok, test_deb=test_deb_ok,
    oof_bge=oof_bge, test_bge=test_bge,
    oof_lex=oof_lex, test_lex=test_lex
)

files.download("submission_clean_4family_blend.csv")
files.download("clean_day1_all_probs.npz")
print("\nSaved & downloaded submission_clean_4family_blend.csv and clean_day1_all_probs.npz!")
print("Test Distribution:\n", sub_4f["emotion"].value_counts())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.34GB            

... (9 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Saved & downloaded submission_clean_4family_blend.csv and clean_day1_all_probs.npz!
Test Distribution:
 emotion
joy         396
anger       347
sadness     278
surprise    188
disgust     120
fear        111
Name: count, dtype: int64


In [ ]:
import os
import re
import json
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_cosine_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report
from google.colab import files

# ==========================================
# 1. Reproducibility & Device
# ==========================================
SEED = 1024  # Fresh seed for Day 2 Large models to maximize ensemble diversity
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ==========================================
# 2. Load Competition Data & Day 1 Saved Probabilities
# ==========================================
with open("final_posts.json", "r", encoding="utf-8") as f:
    raw_posts = json.load(f)

if isinstance(raw_posts, dict):
    if all(isinstance(v, str) for v in list(raw_posts.values())[:5]):
        records = [{"post_id": str(k), "text": v} for k, v in raw_posts.items()]
    else:
        first_val = next(iter(raw_posts.values()))
        raw_posts = first_val if isinstance(first_val, list) else list(raw_posts.values())

if isinstance(raw_posts, list):
    records = []
    for item in raw_posts:
        post = item["root"]["_source"]["post"] if "root" in item else item
        pid = post.get("post_id") or post.get("id")
        txt = post.get("text") or post.get("content") or ""
        records.append({"post_id": str(pid), "text": txt})

posts_df = pd.DataFrame(records)
ident_df = pd.read_csv("data_identification.csv", dtype=str)
emo_df   = pd.read_csv("emotion.csv", dtype=str)
full_df  = posts_df.merge(ident_df, on="post_id", how="inner")

def light_clean(text):
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", "http", text)
    text = re.sub(r"@\w+", "@user", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

full_df["clean_text"] = full_df["text"].apply(light_clean)
train_df = full_df[full_df["set"] == "train"].merge(emo_df, on="post_id", how="inner").reset_index(drop=True)
test_df  = full_df[full_df["set"] == "test"].reset_index(drop=True)

sample_sub = pd.read_csv("sampleSubmission.csv", dtype=str)
sub_id_col = sample_sub.columns[0]
test_df = sample_sub[[sub_id_col]].merge(test_df, left_on=sub_id_col, right_on="post_id", how="left")

le = LabelEncoder()
y_all = le.fit_transform(train_df["label"].astype(str))
classes = le.classes_
num_classes = len(classes)
counts = np.bincount(y_all)

# IMPORTANT: Keep exact same fold split (random_state=42) so all OOF arrays align 1-to-1!
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Load Day 1 clean probabilities
day1 = np.load("clean_day1_all_probs.npz")
oof_rob_base, test_rob_base = day1["oof_rob"], day1["test_rob"]
oof_deb_base, test_deb_base = day1["oof_deb"], day1["test_deb"]
oof_bge, test_bge           = day1["oof_bge"], day1["test_bge"]
oof_lex, test_lex           = day1["oof_lex"], day1["test_lex"]
print("Loaded Day 1 clean probabilities successfully!")

# ==========================================
# 3. Mean+Max Pooling + Multi-Sample Dropout + Layer-wise LR Decay (LLRD)
# ==========================================
class LargeEmotionClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.update({"hidden_dropout_prob": 0.1, "attention_probs_dropout_prob": 0.1})
        self.backbone = AutoModel.from_pretrained(model_name, config=self.config, dtype=torch.float32)
        hidden = self.config.hidden_size
        self.dropouts = nn.ModuleList([nn.Dropout(0.1 * i) for i in range(1, 6)])
        self.fc = nn.Linear(hidden * 2, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state.float()
        mask = attention_mask.unsqueeze(-1).float()
        mean_pool = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        max_pool = out.masked_fill(mask == 0, -1e9).max(dim=1).values
        feat = torch.cat([mean_pool, max_pool], dim=-1)
        logits = sum(self.fc(drop(feat)) for drop in self.dropouts) / len(self.dropouts)
        return logits

def get_llrd_optimizer(model, base_lr=1.5e-5, head_lr=1e-4, weight_decay=0.01, decay=0.95):
    no_decay = ["bias", "LayerNorm.weight", "LayerNorm.bias"]
    num_layers = model.config.num_hidden_layers
    optimizer_grouped_parameters = []

    # Classifier head
    optimizer_grouped_parameters.append({
        "params": [p for n, p in model.fc.named_parameters() if not any(nd in n for nd in no_decay)],
        "weight_decay": weight_decay, "lr": head_lr
    })
    optimizer_grouped_parameters.append({
        "params": [p for n, p in model.fc.named_parameters() if any(nd in n for nd in no_decay)],
        "weight_decay": 0.0, "lr": head_lr
    })

    # Transformer layers (top to bottom)
    layers = model.backbone.encoder.layer
    for layer_idx in range(num_layers - 1, -1, -1):
        lr = base_lr * (decay ** (num_layers - 1 - layer_idx))
        optimizer_grouped_parameters.append({
            "params": [p for n, p in layers[layer_idx].named_parameters() if not any(nd in n for nd in no_decay)],
            "weight_decay": weight_decay, "lr": lr
        })
        optimizer_grouped_parameters.append({
            "params": [p for n, p in layers[layer_idx].named_parameters() if any(nd in n for nd in no_decay)],
            "weight_decay": 0.0, "lr": lr
        })

    # Embeddings & remaining backbone params
    emb_lr = base_lr * (decay ** num_layers)
    assigned = {id(p) for g in optimizer_grouped_parameters for p in g["params"]}
    rem_decay, rem_no_decay = [], []
    for n, p in model.backbone.named_parameters():
        if id(p) not in assigned and p.requires_grad:
            if any(nd in n for nd in no_decay):
                rem_no_decay.append(p)
            else:
                rem_decay.append(p)
    optimizer_grouped_parameters.append({"params": rem_decay, "weight_decay": weight_decay, "lr": emb_lr})
    optimizer_grouped_parameters.append({"params": rem_no_decay, "weight_decay": 0.0, "lr": emb_lr})
    return torch.optim.AdamW(optimizer_grouped_parameters)

def train_large_backbone(model_name, base_lr=1.5e-5, epochs=4, batch_size=16, grad_accum=2, max_len=144, use_fp32=False):
    print(f"\nTraining 5-Fold Clean Large Backbone: {model_name} (LLRD, zero emotion supervision)...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    class TextDS(Dataset):
        def __init__(self, texts, labels=None):
            self.enc = tokenizer(list(texts), truncation=True, max_length=max_len, padding=False)
            self.labels = list(labels) if labels is not None else None
        def __len__(self):
            return len(self.enc["input_ids"])
        def __getitem__(self, i):
            d = {"input_ids": self.enc["input_ids"][i], "attention_mask": self.enc["attention_mask"][i]}
            if self.labels is not None:
                d["labels"] = self.labels[i]
            return d

    def collate(batch):
        ml = max(len(x["input_ids"]) for x in batch)
        pad_id = tokenizer.pad_token_id or 0
        ids, masks, labs = [], [], []
        for x in batch:
            pl = ml - len(x["input_ids"])
            ids.append(x["input_ids"] + [pad_id] * pl)
            masks.append(x["attention_mask"] + [0] * pl)
            if "labels" in x:
                labs.append(x["labels"])
        out = {"input_ids": torch.tensor(ids, dtype=torch.long), "attention_mask": torch.tensor(masks, dtype=torch.long)}
        if labs:
            out["labels"] = torch.tensor(labs, dtype=torch.long)
        return out

    test_loader = DataLoader(TextDS(test_df["clean_text"]), batch_size=64, shuffle=False, collate_fn=collate)
    class_weights = torch.tensor([(len(y_all) / (num_classes * counts[c])) ** 0.45 for c in range(num_classes)], dtype=torch.float32, device=device)

    oof_p = np.zeros((len(train_df), num_classes), dtype=np.float32)
    test_p = np.zeros((len(test_df), num_classes), dtype=np.float32)

    use_bf16 = (not use_fp32) and torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
    use_amp = (not use_fp32) and (device.type == "cuda")

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, y_all), 1):
        t0 = time.time()
        seed_everything(SEED + fold * 17)
        tr_loader = DataLoader(TextDS(train_df.loc[tr_idx, "clean_text"], y_all[tr_idx]), batch_size=batch_size, shuffle=True, collate_fn=collate)
        va_loader = DataLoader(TextDS(train_df.loc[va_idx, "clean_text"], y_all[va_idx]), batch_size=64, shuffle=False, collate_fn=collate)

        model = LargeEmotionClassifier(model_name, num_classes).float().to(device)
        optimizer = get_llrd_optimizer(model, base_lr=base_lr, head_lr=base_lr * 5, weight_decay=0.01, decay=0.95)
        total_steps = (len(tr_loader) // grad_accum) * epochs
        scheduler = get_cosine_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
        scaler = torch.amp.GradScaler("cuda", enabled=(use_amp and not use_bf16))

        best_f1, best_va, best_state = -1.0, None, None
        for ep in range(1, epochs + 1):
            model.train()
            optimizer.zero_grad()
            for step, b in enumerate(tr_loader, 1):
                with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=use_amp):
                    logits = model(b["input_ids"].to(device), b["attention_mask"].to(device))
                    loss = criterion(logits, b["labels"].to(device)) / grad_accum
                if scaler.is_enabled():
                    scaler.scale(loss).backward()
                else:
                    loss.backward()

                if step % grad_accum == 0 or step == len(tr_loader):
                    if scaler.is_enabled():
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

            model.eval()
            va_list = []
            with torch.inference_mode():
                for b in va_loader:
                    with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=use_amp):
                        logits = model(b["input_ids"].to(device), b["attention_mask"].to(device))
                        va_list.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
            va_probs = np.vstack(va_list)
            f1_ep = f1_score(y_all[va_idx], va_probs.argmax(axis=1), average="macro")
            if f1_ep > best_f1:
                best_f1, best_va = f1_ep, va_probs
                best_state = {k: v.cpu() for k, v in model.state_dict().items()}

        oof_p[va_idx] = best_va
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
        model.eval()
        te_list = []
        with torch.inference_mode():
            for b in test_loader:
                with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=use_amp):
                    logits = model(b["input_ids"].to(device), b["attention_mask"].to(device))
                    te_list.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
        test_p += np.vstack(te_list) / 5.0
        print(f"  Fold {fold}/5 | Val Macro-F1: {best_f1:.4f} | {time.time() - t0:.1f}s")

    cv_f1 = f1_score(y_all, oof_p.argmax(axis=1), average="macro")
    print(f"--> {model_name} 5-Fold OOF Macro-F1: {cv_f1:.4f}")
    return oof_p, test_p

# Train both clean Large backbones (RoBERTa-large uses fast bf16/fp16; DeBERTa-v3-large uses safe FP32)
oof_rob_large, test_rob_large = train_large_backbone("FacebookAI/roberta-large", base_lr=1.5e-5, epochs=4, batch_size=16, grad_accum=2, use_fp32=False)
oof_deb_large, test_deb_large = train_large_backbone("microsoft/deberta-v3-large", base_lr=1.2e-5, epochs=4, batch_size=16, grad_accum=2, use_fp32=True)

# ==========================================
# 4. 6-Family Grand Ensemble (Arithmetic, Geometric & OOF-Weighted)
# ==========================================
# 6 completely compliant clean models:
# [0] RoBERTa-large, [1] DeBERTa-v3-large, [2] RoBERTa-base, [3] DeBERTa-v3-base, [4] BGE-Large, [5] Lexical TF-IDF
oofs  = [oof_rob_large, oof_deb_large, oof_rob_base, oof_deb_base, oof_bge, oof_lex]
tests = [test_rob_large, test_deb_large, test_rob_base, test_deb_base, test_bge, test_lex]
names = ["RoBERTa-large", "DeBERTa-v3-large", "RoBERTa-base", "DeBERTa-v3-base", "BGE-Large", "Lexical-TFIDF"]

print("\n" + "=" * 76)
print("INDIVIDUAL CLEAN MODEL 5-FOLD OOF MACRO-F1 SUMMARY")
print("=" * 76)
for n, o in zip(names, oofs):
    print(f"  {n:<20s}: {f1_score(y_all, o.argmax(axis=1), average='macro'):.4f}")

# Candidate A: Balanced 6-Family Arithmetic Blend (Large-heavy)
w_arith = np.array([0.30, 0.25, 0.15, 0.10, 0.12, 0.08])
oof_arith = sum(w * o for w, o in zip(w_arith, oofs))
test_arith = sum(w * t for w, t in zip(w_arith, tests))
f1_arith = f1_score(y_all, oof_arith.argmax(axis=1), average="macro")

# Candidate B: 6-Family Geometric Mean Blend (Log-probability average, strong against overconfident outliers)
eps = 1e-7
oof_geom = np.exp(sum(w * np.log(np.clip(o, eps, 1.0)) for w, o in zip(w_arith, oofs)))
test_geom = np.exp(sum(w * np.log(np.clip(t, eps, 1.0)) for w, t in zip(w_arith, tests)))
f1_geom = f1_score(y_all, oof_geom.argmax(axis=1), average="macro")

# Candidate C: Random Dirichlet OOF Search (3,000 weight combinations, pure raw argmax)
rng = np.random.default_rng(42)
best_dir_w, best_dir_f1 = w_arith, f1_arith
for _ in range(3000):
    w = rng.dirichlet([3.0, 2.5, 1.5, 1.0, 1.2, 0.8])
    s = f1_score(y_all, sum(wi * oi for wi, oi in zip(w, oofs)).argmax(axis=1), average="macro")
    if s > best_dir_f1:
        best_dir_f1, best_dir_w = s, w

oof_dir = sum(w * o for w, o in zip(best_dir_w, oofs))
test_dir = sum(w * t for w, t in zip(best_dir_w, tests))

print("=" * 76)
print("DAY 2 GRAND ENSEMBLE RESULTS (100% COMPLIANT, RAW ARGMAX)")
print("=" * 76)
print(f"  1. RoBERTa-large Single 5-Fold         : {f1_score(y_all, oof_rob_large.argmax(axis=1), average='macro'):.4f}")
print(f"  2. DeBERTa-v3-large Single 5-Fold      : {f1_score(y_all, oof_deb_large.argmax(axis=1), average='macro'):.4f}")
print(f"  3. 6-Family Arithmetic Blend           : {f1_arith:.4f}")
print(f"  4. 6-Family Geometric Mean Blend       : {f1_geom:.4f}")
print(f"  5. 6-Family Dirichlet OOF Optimal      : {best_dir_f1:.4f} | Weights: {np.round(best_dir_w, 3)}")
print("=" * 76)

# Save all 5 Day 2 submission candidates + full probability archive
subs = {
    "sub_day2_1_roberta_large.csv": test_rob_large,
    "sub_day2_2_deberta_large.csv": test_deb_large,
    "sub_day2_3_6family_arith.csv": test_arith,
    "sub_day2_4_6family_geom.csv": test_geom,
    "sub_day2_5_6family_dirichlet.csv": test_dir,
}
for fname, probs in subs.items():
    df_out = pd.DataFrame({"id": test_df[sub_id_col], "emotion": le.inverse_transform(probs.argmax(axis=1))})
    df_out.to_csv(fname, index=False)
    files.download(fname)

np.savez(
    "clean_day2_all_probs.npz",
    oof_rob_large=oof_rob_large, test_rob_large=test_rob_large,
    oof_deb_large=oof_deb_large, test_deb_large=test_deb_large,
    oof_rob_base=oof_rob_base, test_rob_base=test_rob_base,
    oof_deb_base=oof_deb_base, test_deb_base=test_deb_base,
    oof_bge=oof_bge, test_bge=test_bge,
    oof_lex=oof_lex, test_lex=test_lex
)
files.download("clean_day2_all_probs.npz")

Using device: cuda
Loaded Day 1 clean probabilities successfully!

Training 5-Fold Clean Large Backbone: FacebookAI/roberta-large (LLRD, zero emotion supervision)...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (49 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import numpy as np
import pandas as pd
from google.colab import files

# 若檔案已不在暫存區，觸發上傳 clean_day2_all_probs.npz 與 sampleSubmission.csv
needed = [f for f in ["clean_day2_all_probs.npz", "sampleSubmission.csv"] if not os.path.exists(f)]
if needed:
    print("請選擇並上傳:", needed)
    files.upload()

d = np.load("clean_day2_all_probs.npz")
sample_sub = pd.read_csv("sampleSubmission.csv", dtype=str)
sub_id_col = sample_sub.columns[0]
classes = np.array(["anger", "disgust", "fear", "joy", "sadness", "surprise"])

# 5-Family Clean Blend (踢掉欠擬合的 DeBERTa-v3-large)
test_5f = (
    0.35 * d["test_rob_large"]
    + 0.25 * d["test_rob_base"]
    + 0.18 * d["test_bge"]
    + 0.12 * d["test_deb_base"]
    + 0.10 * d["test_lex"]
)

sub_5f = pd.DataFrame({
    "id": sample_sub[sub_id_col],
    "emotion": classes[test_5f.argmax(axis=1)]
})
sub_5f.to_csv("sub_day2_bonus_5family_clean.csv", index=False)
print("Preview:\n", sub_5f.head(3))
print("\nTest Distribution:\n", sub_5f["emotion"].value_counts())
files.download("sub_day2_bonus_5family_clean.csv")

請選擇並上傳: ['clean_day2_all_probs.npz', 'sampleSubmission.csv']


Saving sampleSubmission.csv to sampleSubmission.csv
Saving clean_day2_all_probs.npz to clean_day2_all_probs.npz
Preview:
         id   emotion
0  xyzx9jp  surprise
1  l3vzzxt   sadness
2  g680iug   sadness

Test Distribution:
 emotion
joy         394
anger       342
sadness     273
surprise    186
disgust     129
fear        116
Name: count, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving clean_day2_all_probs.npz to clean_day2_all_probs.npz
Saving data_identification.csv to data_identification.csv
Saving emotion.csv to emotion.csv
Saving final_posts.json to final_posts.json
Saving sampleSubmission.csv to sampleSubmission.csv


In [ ]:
!pip install -q sentence-transformers

import os
import re
import json
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score
from google.colab import files

# ==========================================
# 1. Reproducibility & Device
# ==========================================
SEED = 2026
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
assert device.type == "cuda", "請先切換至 GPU (L4 或 A100) 再執行！"

# ==========================================
# 2. Load Competition Data & Day 2 Saved Probabilities
# ==========================================
with open("final_posts.json", "r", encoding="utf-8") as f:
    raw_posts = json.load(f)

if isinstance(raw_posts, dict):
    if all(isinstance(v, str) for v in list(raw_posts.values())[:5]):
        records = [{"post_id": str(k), "text": v} for k, v in raw_posts.items()]
    else:
        first_val = next(iter(raw_posts.values()))
        raw_posts = first_val if isinstance(first_val, list) else list(raw_posts.values())

if isinstance(raw_posts, list):
    records = []
    for item in raw_posts:
        post = item["root"]["_source"]["post"] if "root" in item else item
        pid = post.get("post_id") or post.get("id")
        txt = post.get("text") or post.get("content") or ""
        records.append({"post_id": str(pid), "text": txt})

posts_df = pd.DataFrame(records)
ident_df = pd.read_csv("data_identification.csv", dtype=str)
emo_df   = pd.read_csv("emotion.csv", dtype=str)
full_df  = posts_df.merge(ident_df, on="post_id", how="inner")

def light_clean(text):
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", "http", text)
    text = re.sub(r"@\w+", "@user", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

full_df["clean_text"] = full_df["text"].apply(light_clean)
train_df = full_df[full_df["set"] == "train"].merge(emo_df, on="post_id", how="inner").reset_index(drop=True)
test_df  = full_df[full_df["set"] == "test"].reset_index(drop=True)

sample_sub = pd.read_csv("sampleSubmission.csv", dtype=str)
sub_id_col = sample_sub.columns[0]
test_df = sample_sub[[sub_id_col]].merge(test_df, left_on=sub_id_col, right_on="post_id", how="left")

le = LabelEncoder()
y_all = le.fit_transform(train_df["label"].astype(str))
classes = le.classes_
num_classes = len(classes)
counts = np.bincount(y_all)

# Keep exact same 5-fold split (random_state=42) for 1-to-1 OOF alignment
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Load Day 2 probabilities
d2 = np.load("clean_day2_all_probs.npz")
oof_rob_large, test_rob_large = d2["oof_rob_large"], d2["test_rob_large"]
oof_deb_large_old, test_deb_large_old = d2["oof_deb_large"], d2["test_deb_large"]
oof_rob_base, test_rob_base   = d2["oof_rob_base"], d2["test_rob_base"]
oof_deb_base, test_deb_base   = d2["oof_deb_base"], d2["test_deb_base"]
oof_bge, test_bge             = d2["oof_bge"], d2["test_bge"]
oof_lex, test_lex             = d2["oof_lex"], d2["test_lex"]
print("Loaded Day 2 clean probabilities successfully!")

# ==========================================
# 3. Model Family #7: Clean intfloat/e5-large-v2 (MIT, zero emotion supervision)
# ==========================================
print("\n[1/2] Extracting Clean E5-Large-v2 Embeddings (zero emotion supervision)...")
e5_model = SentenceTransformer("intfloat/e5-large-v2", device=str(device))
# E5 expects "passage: " prefix for document/post encoding
tr_e5_texts = ["passage: " + t for t in train_df["clean_text"].tolist()]
te_e5_texts = ["passage: " + t for t in test_df["clean_text"].tolist()]

X_tr_e5 = normalize(e5_model.encode(tr_e5_texts, batch_size=64, show_progress_bar=False))
X_te_e5 = normalize(e5_model.encode(te_e5_texts, batch_size=64, show_progress_bar=False))

tempered_w = {c: (len(y_all) / (num_classes * counts[c])) ** 0.45 for c in range(num_classes)}
oof_e5 = np.zeros((len(train_df), num_classes), dtype=np.float32)
test_e5 = np.zeros((len(test_df), num_classes), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr_e5, y_all)):
    m_svc = CalibratedClassifierCV(LinearSVC(C=0.35, class_weight=tempered_w, random_state=SEED + fold), cv=3)
    m_lr  = LogisticRegression(C=2.0, class_weight=tempered_w, max_iter=1000, random_state=SEED + fold)
    m_svc.fit(X_tr_e5[tr_idx], y_all[tr_idx])
    m_lr.fit(X_tr_e5[tr_idx], y_all[tr_idx])
    oof_e5[va_idx] = 0.5 * m_svc.predict_proba(X_tr_e5[va_idx]) + 0.5 * m_lr.predict_proba(X_tr_e5[va_idx])
    test_e5 += (0.5 * m_svc.predict_proba(X_te_e5) + 0.5 * m_lr.predict_proba(X_te_e5)) / 5.0

print(f"--> Clean E5-Large-v2 5-Fold OOF Macro-F1: {f1_score(y_all, oof_e5.argmax(axis=1), average='macro'):.4f}")
del e5_model
torch.cuda.empty_cache()

# ==========================================
# 4. Fixed DeBERTa-v3-large (Standard Classification Head, Uniform LR=1.5e-5, Safe FP32)
# ==========================================
def train_fixed_deberta_large(model_name="microsoft/deberta-v3-large", lr=1.5e-5, epochs=4, batch_size=16, grad_accum=2, max_len=128):
    print(f"\n[2/2] Training Fixed 5-Fold {model_name} (Standard Head, Uniform LR={lr}, FP32)...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    class TextDS(Dataset):
        def __init__(self, texts, labels=None):
            self.enc = tokenizer(list(texts), truncation=True, max_length=max_len, padding=False)
            self.labels = list(labels) if labels is not None else None
        def __len__(self):
            return len(self.enc["input_ids"])
        def __getitem__(self, i):
            d = {"input_ids": self.enc["input_ids"][i], "attention_mask": self.enc["attention_mask"][i]}
            if self.labels is not None:
                d["labels"] = self.labels[i]
            return d

    def collate(batch):
        ml = max(len(x["input_ids"]) for x in batch)
        pad_id = tokenizer.pad_token_id or 0
        ids, masks, labs = [], [], []
        for x in batch:
            pl = ml - len(x["input_ids"])
            ids.append(x["input_ids"] + [pad_id] * pl)
            masks.append(x["attention_mask"] + [0] * pl)
            if "labels" in x:
                labs.append(x["labels"])
        out = {"input_ids": torch.tensor(ids, dtype=torch.long), "attention_mask": torch.tensor(masks, dtype=torch.long)}
        if labs:
            out["labels"] = torch.tensor(labs, dtype=torch.long)
        return out

    test_loader = DataLoader(TextDS(test_df["clean_text"]), batch_size=64, shuffle=False, collate_fn=collate)
    class_weights = torch.tensor([(len(y_all) / (num_classes * counts[c])) ** 0.45 for c in range(num_classes)], dtype=torch.float32, device=device)

    oof_p = np.zeros((len(train_df), num_classes), dtype=np.float32)
    test_p = np.zeros((len(test_df), num_classes), dtype=np.float32)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, y_all), 1):
        t0 = time.time()
        seed_everything(SEED + fold * 13)
        tr_loader = DataLoader(TextDS(train_df.loc[tr_idx, "clean_text"], y_all[tr_idx]), batch_size=batch_size, shuffle=True, collate_fn=collate)
        va_loader = DataLoader(TextDS(train_df.loc[va_idx, "clean_text"], y_all[va_idx]), batch_size=64, shuffle=False, collate_fn=collate)

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=num_classes, hidden_dropout_prob=0.1, attention_probs_dropout_prob=0.1, dtype=torch.float32
        ).float().to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        total_steps = (len(tr_loader) // grad_accum) * epochs
        scheduler = get_cosine_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)

        best_f1, best_va, best_state = -1.0, None, None
        for ep in range(1, epochs + 1):
            model.train()
            optimizer.zero_grad()
            for step, b in enumerate(tr_loader, 1):
                logits = model(input_ids=b["input_ids"].to(device), attention_mask=b["attention_mask"].to(device)).logits.float()
                loss = criterion(logits, b["labels"].to(device)) / grad_accum
                loss.backward()
                if step % grad_accum == 0 or step == len(tr_loader):
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

            model.eval()
            va_list = []
            with torch.inference_mode():
                for b in va_loader:
                    logits = model(input_ids=b["input_ids"].to(device), attention_mask=b["attention_mask"].to(device)).logits.float()
                    va_list.append(torch.softmax(logits, dim=-1).cpu().numpy())
            va_probs = np.vstack(va_list)
            f1_ep = f1_score(y_all[va_idx], va_probs.argmax(axis=1), average="macro")
            if f1_ep > best_f1:
                best_f1, best_va = f1_ep, va_probs
                best_state = {k: v.cpu() for k, v in model.state_dict().items()}

        oof_p[va_idx] = best_va
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
        model.eval()
        te_list = []
        with torch.inference_mode():
            for b in test_loader:
                logits = model(input_ids=b["input_ids"].to(device), attention_mask=b["attention_mask"].to(device)).logits.float()
                te_list.append(torch.softmax(logits, dim=-1).cpu().numpy())
        test_p += np.vstack(te_list) / 5.0
        print(f"  Fold {fold}/5 | Val Macro-F1: {best_f1:.4f} | {time.time() - t0:.1f}s")

    print(f"--> Fixed {model_name} 5-Fold OOF Macro-F1: {f1_score(y_all, oof_p.argmax(axis=1), average='macro'):.4f}")
    return oof_p, test_p

oof_deb_large_fix, test_deb_large_fix = train_fixed_deberta_large()

# ==========================================
# 5. Day 3 7-Family Grand Ensemble (No Overfitting!)
# ==========================================
# [0] RoBERTa-large, [1] DeBERTa-v3-large (Fixed), [2] RoBERTa-base,
# [3] DeBERTa-v3-base, [4] BGE-Large, [5] E5-Large-v2, [6] Lexical-TFIDF
oofs  = [oof_rob_large, oof_deb_large_fix, oof_rob_base, oof_deb_base, oof_bge, oof_e5, oof_lex]
tests = [test_rob_large, test_deb_large_fix, test_rob_base, test_deb_base, test_bge, test_e5, test_lex]
names = ["RoBERTa-large", "DeBERTa-v3-large(Fix)", "RoBERTa-base", "DeBERTa-v3-base", "BGE-Large", "E5-Large-v2", "Lexical-TFIDF"]

print("\n" + "=" * 78)
print("DAY 3 INDIVIDUAL CLEAN MODEL 5-FOLD OOF MACRO-F1 SUMMARY")
print("=" * 78)
for n, o in zip(names, oofs):
    print(f"  {n:<24s}: {f1_score(y_all, o.argmax(axis=1), average='macro'):.4f}")

# Candidate 2: Balanced 7-Family Arithmetic Blend
w_bal = np.array([0.24, 0.24, 0.14, 0.08, 0.11, 0.11, 0.08])
oof_bal = sum(w * o for w, o in zip(w_bal, oofs))
test_bal = sum(w * t for w, t in zip(w_bal, tests))

# Candidate 3: Large-Heavy 7-Family Arithmetic Blend (RoBERTa-L + DeB-L = 56%)
w_lrg = np.array([0.28, 0.28, 0.12, 0.06, 0.10, 0.10, 0.06])
oof_lrg = sum(w * o for w, o in zip(w_lrg, oofs))
test_lrg = sum(w * t for w, t in zip(w_lrg, tests))

# Candidate 4: 7-Family Geometric Mean Blend (Log-Probability Average)
eps = 1e-7
oof_geom = np.exp(sum(w * np.log(np.clip(o, eps, 1.0)) for w, o in zip(w_bal, oofs)))
test_geom = np.exp(sum(w * np.log(np.clip(t, eps, 1.0)) for w, t in zip(w_bal, tests)))

# Candidate 5: Strongly Regularized L2 LogisticRegression Meta-Stacker (Strict 5-Fold Nested OOF)
X_meta_tr = np.hstack(oofs)
X_meta_te = np.hstack(tests)
oof_meta = np.zeros((len(train_df), num_classes), dtype=np.float32)
test_meta = np.zeros((len(test_df), num_classes), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_meta_tr, y_all)):
    meta_clf = LogisticRegression(C=0.1, max_iter=1000, random_state=SEED + fold)
    meta_clf.fit(X_meta_tr[tr_idx], y_all[tr_idx])
    oof_meta[va_idx] = meta_clf.predict_proba(X_meta_tr[va_idx])
    test_meta += meta_clf.predict_proba(X_meta_te) / 5.0

print("=" * 78)
print("DAY 3 GRAND ENSEMBLE RESULTS (100% COMPLIANT, ZERO OVERFITTING)")
print("=" * 78)
print(f"  1. Fixed DeBERTa-v3-large Single 5-Fold : {f1_score(y_all, oof_deb_large_fix.argmax(axis=1), average='macro'):.4f}")
print(f"  2. 7-Family Balanced Arithmetic Blend   : {f1_score(y_all, oof_bal.argmax(axis=1), average='macro'):.4f}")
print(f"  3. 7-Family Large-Heavy Arith Blend     : {f1_score(y_all, oof_lrg.argmax(axis=1), average='macro'):.4f}")
print(f"  4. 7-Family Geometric Mean Blend        : {f1_score(y_all, oof_geom.argmax(axis=1), average='macro'):.4f}")
print(f"  5. 7-Family L2 Meta-Stacker (C=0.1)     : {f1_score(y_all, oof_meta.argmax(axis=1), average='macro'):.4f}")
print("=" * 78)

subs = {
    "sub_day3_1_deberta_large_fixed.csv": test_deb_large_fix,
    "sub_day3_2_7family_arith_balanced.csv": test_bal,
    "sub_day3_3_7family_large_heavy.csv": test_lrg,
    "sub_day3_4_7family_geom.csv": test_geom,
    "sub_day3_5_7family_meta_l2.csv": test_meta,
}
for fname, probs in subs.items():
    df_out = pd.DataFrame({"id": test_df[sub_id_col], "emotion": le.inverse_transform(probs.argmax(axis=1))})
    df_out.to_csv(fname, index=False)
    files.download(fname)

np.savez(
    "clean_day3_all_probs.npz",
    oof_rob_large=oof_rob_large, test_rob_large=test_rob_large,
    oof_deb_large_fix=oof_deb_large_fix, test_deb_large_fix=test_deb_large_fix,
    oof_deb_large_old=oof_deb_large_old, test_deb_large_old=test_deb_large_old,
    oof_rob_base=oof_rob_base, test_rob_base=test_rob_base,
    oof_deb_base=oof_deb_base, test_deb_base=test_deb_base,
    oof_bge=oof_bge, test_bge=test_bge,
    oof_e5=oof_e5, test_e5=test_e5,
    oof_lex=oof_lex, test_lex=test_lex
)
files.download("clean_day3_all_probs.npz")

Using device: cuda
Loaded Day 2 clean probabilities successfully!

[1/2] Extracting Clean E5-Large-v2 Embeddings (zero emotion supervision)...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/67.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.34GB            

... (40 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from google.colab import files

# 改用五折全正常的 Day 2 DeBERTa-v3-large + 新加入的 E5-Large-v2 (7-Family 完整健康版)
oofs_ok  = [oof_rob_large, oof_deb_large_old, oof_rob_base, oof_deb_base, oof_bge, oof_e5, oof_lex]
tests_ok = [test_rob_large, test_deb_large_old, test_rob_base, test_deb_base, test_bge, test_e5, test_lex]

# 1. 7-Family Arithmetic Blend (沿用昨天 0.6127 的黃金比例，將 BGE 拆成 BGE + E5)
w_arith_v1 = np.array([0.28, 0.22, 0.14, 0.08, 0.10, 0.10, 0.08])
oof_v1 = sum(w * o for w, o in zip(w_arith_v1, oofs_ok))
test_v1 = sum(w * t for w, t in zip(w_arith_v1, tests_ok))

# 2. 7-Family RoBERTa-Heavy Arithmetic Blend (加重最強單模 RoBERTa-L 32%)
w_arith_v2 = np.array([0.32, 0.20, 0.14, 0.08, 0.10, 0.10, 0.06])
oof_v2 = sum(w * o for w, o in zip(w_arith_v2, oofs_ok))
test_v2 = sum(w * t for w, t in zip(w_arith_v2, tests_ok))

# 3. 7-Family Geometric Mean Blend (Log-Probability Average)
eps = 1e-7
oof_v3 = np.exp(sum(w * np.log(np.clip(o, eps, 1.0)) for w, o in zip(w_arith_v1, oofs_ok)))
test_v3 = np.exp(sum(w * np.log(np.clip(t, eps, 1.0)) for w, t in zip(w_arith_v1, tests_ok)))

# 4 & 5. 7-Family L2 Meta-Stacker (C=0.1 與更強正則化 C=0.05，嚴格 5-Fold Nested OOF)
X_tr_ok = np.hstack(oofs_ok)
X_te_ok = np.hstack(tests_ok)

def run_meta(C_val):
    oof_m = np.zeros((len(train_df), num_classes), dtype=np.float32)
    test_m = np.zeros((len(test_df), num_classes), dtype=np.float32)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr_ok, y_all)):
        clf = LogisticRegression(C=C_val, max_iter=1000, random_state=42 + fold)
        clf.fit(X_tr_ok[tr_idx], y_all[tr_idx])
        oof_m[va_idx] = clf.predict_proba(X_tr_ok[va_idx])
        test_m += clf.predict_proba(X_te_ok) / 5.0
    return oof_m, test_m

oof_v4, test_v4 = run_meta(0.10)
oof_v5, test_v5 = run_meta(0.05)

print("=" * 76)
print("HEALTHY 7-FAMILY DAY 3 ENSEMBLE SUMMARY (100% COMPLIANT)")
print("=" * 76)
print(f"  1. 7-Family Arith Balanced (v1)   : {f1_score(y_all, oof_v1.argmax(axis=1), average='macro'):.4f}")
print(f"  2. 7-Family Arith RoBERTa-Heavy   : {f1_score(y_all, oof_v2.argmax(axis=1), average='macro'):.4f}")
print(f"  3. 7-Family Geometric Mean        : {f1_score(y_all, oof_v3.argmax(axis=1), average='macro'):.4f}")
print(f"  4. 7-Family L2 Meta-Stacker C=0.1 : {f1_score(y_all, oof_v4.argmax(axis=1), average='macro'):.4f}")
print(f"  5. 7-Family L2 Meta-Stacker C=0.05: {f1_score(y_all, oof_v5.argmax(axis=1), average='macro'):.4f}")
print("=" * 76)

subs_day3_ok = {
    "sub_day3_ok_1_7fam_arith_bal.csv": test_v1,
    "sub_day3_ok_2_7fam_arith_robheavy.csv": test_v2,
    "sub_day3_ok_3_7fam_geom.csv": test_v3,
    "sub_day3_ok_4_7fam_meta_c010.csv": test_v4,
    "sub_day3_ok_5_7fam_meta_c005.csv": test_v5,
}
for fname, probs in subs_day3_ok.items():
    pd.DataFrame({"id": test_df[sub_id_col], "emotion": le.inverse_transform(probs.argmax(axis=1))}).to_csv(fname, index=False)
    files.download(fname)

HEALTHY 7-FAMILY DAY 3 ENSEMBLE SUMMARY (100% COMPLIANT)
  1. 7-Family Arith Balanced (v1)   : 0.6175
  2. 7-Family Arith RoBERTa-Heavy   : 0.6155
  3. 7-Family Geometric Mean        : 0.6190
  4. 7-Family L2 Meta-Stacker C=0.1 : 0.6190
  5. 7-Family L2 Meta-Stacker C=0.05: 0.6182


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving clean_day3_all_probs.npz to clean_day3_all_probs.npz
Saving data_identification.csv to data_identification.csv
Saving emotion.csv to emotion.csv
Saving final_posts.json to final_posts.json
Saving sampleSubmission.csv to sampleSubmission.csv


In [ ]:
!pip install -q sentence-transformers

import os
import re
import json
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_cosine_schedule_with_warmup
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score
from google.colab import files

# ==========================================
# 1. Reproducibility & Device
# ==========================================
SEED = 777  # Second independent seed for RoBERTa-large diversity
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
assert device.type == "cuda", "請先切換至 GPU (L4 或 A100) 再執行！"

# ==========================================
# 2. Load Competition Data & Day 3 Saved Probabilities
# ==========================================
with open("final_posts.json", "r", encoding="utf-8") as f:
    raw_posts = json.load(f)

if isinstance(raw_posts, dict):
    if all(isinstance(v, str) for v in list(raw_posts.values())[:5]):
        records = [{"post_id": str(k), "text": v} for k, v in raw_posts.items()]
    else:
        first_val = next(iter(raw_posts.values()))
        raw_posts = first_val if isinstance(first_val, list) else list(raw_posts.values())

if isinstance(raw_posts, list):
    records = []
    for item in raw_posts:
        post = item["root"]["_source"]["post"] if "root" in item else item
        pid = post.get("post_id") or post.get("id")
        txt = post.get("text") or post.get("content") or ""
        records.append({"post_id": str(pid), "text": txt})

posts_df = pd.DataFrame(records)
ident_df = pd.read_csv("data_identification.csv", dtype=str)
emo_df   = pd.read_csv("emotion.csv", dtype=str)
full_df  = posts_df.merge(ident_df, on="post_id", how="inner")

def light_clean(text):
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", "http", text)
    text = re.sub(r"@\w+", "@user", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

full_df["clean_text"] = full_df["text"].apply(light_clean)
train_df = full_df[full_df["set"] == "train"].merge(emo_df, on="post_id", how="inner").reset_index(drop=True)
test_df  = full_df[full_df["set"] == "test"].reset_index(drop=True)

sample_sub = pd.read_csv("sampleSubmission.csv", dtype=str)
sub_id_col = sample_sub.columns[0]
test_df = sample_sub[[sub_id_col]].merge(test_df, left_on=sub_id_col, right_on="post_id", how="left")

le = LabelEncoder()
y_all = le.fit_transform(train_df["label"].astype(str))
classes = le.classes_
num_classes = len(classes)
counts = np.bincount(y_all)

# Exact same 5-fold split (random_state=42) for 1-to-1 OOF alignment
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Load Day 3 probabilities (7 healthy families)
d3 = np.load("clean_day3_all_probs.npz")
oof_rob_l1, test_rob_l1 = d3["oof_rob_large"], d3["test_rob_large"]
oof_deb_l,  test_deb_l  = d3["oof_deb_large_old"], d3["test_deb_large_old"]
oof_rob_b,  test_rob_b  = d3["oof_rob_base"], d3["test_rob_base"]
oof_deb_b,  test_deb_b  = d3["oof_deb_base"], d3["test_deb_base"]
oof_bge,    test_bge    = d3["oof_bge"], d3["test_bge"]
oof_e5,     test_e5     = d3["oof_e5"], d3["test_e5"]
oof_lex,    test_lex    = d3["oof_lex"], d3["test_lex"]
print("Loaded Day 3 clean probabilities successfully!")

# ==========================================
# 3. Model Family #8: Clean all-mpnet-base-v2 (Apache 2.0, zero emotion supervision)
# ==========================================
print("\n[1/2] Extracting Clean all-mpnet-base-v2 Embeddings (zero emotion supervision)...")
mpnet = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=str(device))
X_tr_mp = normalize(mpnet.encode(train_df["clean_text"].tolist(), batch_size=64, show_progress_bar=False))
X_te_mp = normalize(mpnet.encode(test_df["clean_text"].tolist(), batch_size=64, show_progress_bar=False))

tempered_w = {c: (len(y_all) / (num_classes * counts[c])) ** 0.45 for c in range(num_classes)}
oof_mpnet = np.zeros((len(train_df), num_classes), dtype=np.float32)
test_mpnet = np.zeros((len(test_df), num_classes), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr_mp, y_all)):
    m_svc = CalibratedClassifierCV(LinearSVC(C=0.35, class_weight=tempered_w, random_state=SEED + fold), cv=3)
    m_lr  = LogisticRegression(C=2.0, class_weight=tempered_w, max_iter=1000, random_state=SEED + fold)
    m_svc.fit(X_tr_mp[tr_idx], y_all[tr_idx])
    m_lr.fit(X_tr_mp[tr_idx], y_all[tr_idx])
    oof_mpnet[va_idx] = 0.5 * m_svc.predict_proba(X_tr_mp[va_idx]) + 0.5 * m_lr.predict_proba(X_tr_mp[va_idx])
    test_mpnet += (0.5 * m_svc.predict_proba(X_te_mp) + 0.5 * m_lr.predict_proba(X_te_mp)) / 5.0

print(f"--> Clean all-mpnet-base-v2 5-Fold OOF Macro-F1: {f1_score(y_all, oof_mpnet.argmax(axis=1), average='macro'):.4f}")
del mpnet
torch.cuda.empty_cache()

# ==========================================
# 4. Model Family #9: Clean RoBERTa-large Seed 2 (LLRD + Mean/Max Pool + 5x MSD)
# ==========================================
class MeanMaxClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.update({"hidden_dropout_prob": 0.1, "attention_probs_dropout_prob": 0.1})
        self.backbone = AutoModel.from_pretrained(model_name, config=self.config)
        self.dropouts = nn.ModuleList([nn.Dropout(0.1 * i) for i in range(1, 6)])
        self.fc = nn.Linear(self.config.hidden_size * 2, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        mean_pool = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        max_pool  = out.masked_fill(mask == 0, -1e4).max(dim=1).values
        feat = torch.cat([mean_pool, max_pool], dim=-1)
        return sum(self.fc(d(feat)) for d in self.dropouts) / len(self.dropouts)

def get_llrd_params(model, base_lr=2e-5, head_lr=1e-4, llrd=0.95, wd=0.01):
    no_decay = ["bias", "LayerNorm.bias", "LayerNorm.weight"]
    groups = [{"params": [p for n, p in model.fc.named_parameters()], "lr": head_lr, "weight_decay": wd}]
    num_layers = model.config.num_hidden_layers
    for name, p in model.backbone.named_parameters():
        if not p.requires_grad:
            continue
        decay = 0.0 if any(nd in name for nd in no_decay) else wd
        if "embeddings" in name:
            lr = base_lr * (llrd ** (num_layers + 1))
        elif "encoder.layer." in name:
            layer_idx = int(name.split("encoder.layer.")[1].split(".")[0])
            lr = base_lr * (llrd ** (num_layers - layer_idx))
        else:
            lr = base_lr
        groups.append({"params": [p], "lr": lr, "weight_decay": decay})
    return groups

print("\n[2/2] Training 5-Fold Clean RoBERTa-large (Seed 2, LLRD, bf16)...")
model_name = "FacebookAI/roberta-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)

class TextDS(Dataset):
    def __init__(self, texts, labels=None):
        self.enc = tokenizer(list(texts), truncation=True, max_length=128, padding=False)
        self.labels = list(labels) if labels is not None else None
    def __len__(self):
        return len(self.enc["input_ids"])
    def __getitem__(self, i):
        d = {"input_ids": self.enc["input_ids"][i], "attention_mask": self.enc["attention_mask"][i]}
        if self.labels is not None:
            d["labels"] = self.labels[i]
        return d

def collate(batch):
    ml = max(len(x["input_ids"]) for x in batch)
    pad_id = tokenizer.pad_token_id or 0
    ids, masks, labs = [], [], []
    for x in batch:
        pl = ml - len(x["input_ids"])
        ids.append(x["input_ids"] + [pad_id] * pl)
        masks.append(x["attention_mask"] + [0] * pl)
        if "labels" in x:
            labs.append(x["labels"])
    out = {"input_ids": torch.tensor(ids, dtype=torch.long), "attention_mask": torch.tensor(masks, dtype=torch.long)}
    if labs:
        out["labels"] = torch.tensor(labs, dtype=torch.long)
    return out

test_loader = DataLoader(TextDS(test_df["clean_text"]), batch_size=64, shuffle=False, collate_fn=collate)
class_weights = torch.tensor([(len(y_all) / (num_classes * counts[c])) ** 0.45 for c in range(num_classes)], dtype=torch.float32, device=device)

oof_rob_l2 = np.zeros((len(train_df), num_classes), dtype=np.float32)
test_rob_l2 = np.zeros((len(test_df), num_classes), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, y_all), 1):
    t0 = time.time()
    seed_everything(SEED + fold * 19)
    tr_loader = DataLoader(TextDS(train_df.loc[tr_idx, "clean_text"], y_all[tr_idx]), batch_size=16, shuffle=True, collate_fn=collate)
    va_loader = DataLoader(TextDS(train_df.loc[va_idx, "clean_text"], y_all[va_idx]), batch_size=64, shuffle=False, collate_fn=collate)

    model = MeanMaxClassifier(model_name, num_classes).to(device)
    optimizer = torch.optim.AdamW(get_llrd_params(model))
    epochs, grad_accum = 4, 2
    total_steps = (len(tr_loader) // grad_accum) * epochs
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)

    best_f1, best_va, best_state = -1.0, None, None
    for ep in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        for step, b in enumerate(tr_loader, 1):
            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                logits = model(b["input_ids"].to(device), b["attention_mask"].to(device))
                loss = criterion(logits.float(), b["labels"].to(device)) / grad_accum
            loss.backward()
            if step % grad_accum == 0 or step == len(tr_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

        model.eval()
        va_list = []
        with torch.inference_mode():
            for b in va_loader:
                with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                    logits = model(b["input_ids"].to(device), b["attention_mask"].to(device))
                va_list.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
        va_probs = np.vstack(va_list)
        f1_ep = f1_score(y_all[va_idx], va_probs.argmax(axis=1), average="macro")
        if f1_ep > best_f1:
            best_f1, best_va = f1_ep, va_probs
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}

    oof_rob_l2[va_idx] = best_va
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    model.eval()
    te_list = []
    with torch.inference_mode():
        for b in test_loader:
            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                logits = model(b["input_ids"].to(device), b["attention_mask"].to(device))
            te_list.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    test_rob_l2 += np.vstack(te_list) / 5.0
    print(f"  Fold {fold}/5 | Val Macro-F1: {best_f1:.4f} | {time.time() - t0:.1f}s")

print(f"--> Clean RoBERTa-large (Seed 2) 5-Fold OOF Macro-F1: {f1_score(y_all, oof_rob_l2.argmax(axis=1), average='macro'):.4f}")

# ==========================================
# 5. Strict 5-Fold Nested CV Class Multiplier Optimizer
# ==========================================
def fit_multipliers(probs, y_true, n_rounds=3):
    mult = np.ones(probs.shape[1], dtype=np.float32)
    best_score = f1_score(y_true, (probs * mult).argmax(axis=1), average="macro")
    grid = np.linspace(0.75, 1.35, 25)
    for _ in range(n_rounds):
        for c in range(probs.shape[1]):
            cur_best_m = mult[c]
            for g in grid:
                mult[c] = g
                sc = f1_score(y_true, (probs * mult).argmax(axis=1), average="macro")
                if sc > best_score + 1e-6:
                    best_score = sc
                    cur_best_m = g
            mult[c] = cur_best_m
    return mult

def nested_cv_multipliers(oof_probs, test_probs, y_true, damping=0.5):
    oof_full = np.zeros_like(oof_probs)
    oof_damp = np.zeros_like(oof_probs)
    fold_mults = []
    for tr_idx, va_idx in skf.split(oof_probs, y_true):
        m = fit_multipliers(oof_probs[tr_idx], y_true[tr_idx])
        fold_mults.append(m)
        m_damp = damping * m + (1.0 - damping) * 1.0
        oof_full[va_idx] = oof_probs[va_idx] * m
        oof_damp[va_idx] = oof_probs[va_idx] * m_damp
    avg_m = np.mean(fold_mults, axis=0)
    avg_m_damp = damping * avg_m + (1.0 - damping) * 1.0
    return (
        oof_full, test_probs * avg_m, avg_m,
        oof_damp, test_probs * avg_m_damp, avg_m_damp
    )

# ==========================================
# 6. Day 4 Final Ensembles (7-Family & 9-Family)
# ==========================================
# Day 3 Champion 7-Family Arith
w7 = np.array([0.28, 0.22, 0.14, 0.08, 0.10, 0.10, 0.08])
oofs7  = [oof_rob_l1, oof_deb_l, oof_rob_b, oof_deb_b, oof_bge, oof_e5, oof_lex]
tests7 = [test_rob_l1, test_deb_l, test_rob_b, test_deb_b, test_bge, test_e5, test_lex]
oof_7f_raw  = sum(w * o for w, o in zip(w7, oofs7))
test_7f_raw = sum(w * t for w, t in zip(w7, tests7))
_, _, _, oof_7f_damp, test_7f_damp, m7_damp = nested_cv_multipliers(oof_7f_raw, test_7f_raw, y_all, damping=0.5)

# Day 4 Grand 9-Family Arith
# [0] RoB-L1, [1] RoB-L2, [2] DeB-L, [3] RoB-B, [4] DeB-B, [5] BGE, [6] E5, [7] MPNet, [8] Lex
w9 = np.array([0.20, 0.20, 0.18, 0.10, 0.06, 0.07, 0.07, 0.06, 0.06])
oofs9  = [oof_rob_l1, oof_rob_l2, oof_deb_l, oof_rob_b, oof_deb_b, oof_bge, oof_e5, oof_mpnet, oof_lex]
tests9 = [test_rob_l1, test_rob_l2, test_deb_l, test_rob_b, test_deb_b, test_bge, test_e5, test_mpnet, test_lex]
oof_9f_raw  = sum(w * o for w, o in zip(w9, oofs9))
test_9f_raw = sum(w * t for w, t in zip(w9, tests9))
oof_9f_full, test_9f_full, m9_full, oof_9f_damp, test_9f_damp, m9_damp = nested_cv_multipliers(oof_9f_raw, test_9f_raw, y_all, damping=0.5)

# 9-Family L2 Meta-Stacker (C=0.10)
X_tr_9 = np.hstack(oofs9)
X_te_9 = np.hstack(tests9)
oof_9f_meta = np.zeros((len(train_df), num_classes), dtype=np.float32)
test_9f_meta = np.zeros((len(test_df), num_classes), dtype=np.float32)
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr_9, y_all)):
    clf = LogisticRegression(C=0.10, max_iter=1000, random_state=42 + fold)
    clf.fit(X_tr_9[tr_idx], y_all[tr_idx])
    oof_9f_meta[va_idx] = clf.predict_proba(X_tr_9[va_idx])
    test_9f_meta += clf.predict_proba(X_te_9) / 5.0

print("\n" + "=" * 78)
print("DAY 4 FINAL 9-FAMILY ENSEMBLE SUMMARY (100% COMPLIANT)")
print("=" * 78)
print(f"  [Ref] Day 3 7-Family Raw Arith          : {f1_score(y_all, oof_7f_raw.argmax(axis=1), average='macro'):.4f}")
print(f"  1. 7-Family Arith + Damped Mult (0.5x)  : {f1_score(y_all, oof_7f_damp.argmax(axis=1), average='macro'):.4f} | Mult: {np.round(m7_damp, 3)}")
print(f"  2. 9-Family Arith Raw Argmax            : {f1_score(y_all, oof_9f_raw.argmax(axis=1), average='macro'):.4f}")
print(f"  3. 9-Family Arith + Damped Mult (0.5x)  : {f1_score(y_all, oof_9f_damp.argmax(axis=1), average='macro'):.4f} | Mult: {np.round(m9_damp, 3)}")
print(f"  4. 9-Family Arith + Full Nested Mult    : {f1_score(y_all, oof_9f_full.argmax(axis=1), average='macro'):.4f} | Mult: {np.round(m9_full, 3)}")
print(f"  5. 9-Family L2 Meta-Stacker (C=0.10)    : {f1_score(y_all, oof_9f_meta.argmax(axis=1), average='macro'):.4f}")
print("=" * 78)

subs_day4 = {
    "sub_day4_1_7fam_arith_damped_mult.csv": test_7f_damp,
    "sub_day4_2_9fam_arith_raw.csv": test_9f_raw,
    "sub_day4_3_9fam_arith_damped_mult.csv": test_9f_damp,
    "sub_day4_4_9fam_arith_full_mult.csv": test_9f_full,
    "sub_day4_5_9fam_meta_l2.csv": test_9f_meta,
}
for fname, probs in subs_day4.items():
    pd.DataFrame({"id": test_df[sub_id_col], "emotion": le.inverse_transform(probs.argmax(axis=1))}).to_csv(fname, index=False)
    files.download(fname)

np.savez(
    "clean_day4_all_probs.npz",
    oof_rob_l1=oof_rob_l1, test_rob_l1=test_rob_l1,
    oof_rob_l2=oof_rob_l2, test_rob_l2=test_rob_l2,
    oof_deb_l=oof_deb_l,   test_deb_l=test_deb_l,
    oof_rob_b=oof_rob_b,   test_rob_b=test_rob_b,
    oof_deb_b=oof_deb_b,   test_deb_b=test_deb_b,
    oof_bge=oof_bge,       test_bge=test_bge,
    oof_e5=oof_e5,         test_e5=test_e5,
    oof_mpnet=oof_mpnet,   test_mpnet=test_mpnet,
    oof_lex=oof_lex,       test_lex=test_lex,
)
files.download("clean_day4_all_probs.npz")

Using device: cuda
Loaded Day 3 clean probabilities successfully!

[1/2] Extracting Clean all-mpnet-base-v2 Embeddings (zero emotion supervision)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

... (39 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>